# Station 02｜RNN：災害應變訊息智慧分流

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/day6_ai_solution_lab/02_rnn_disaster_triage.ipynb)

**客戶任務：** 判斷短訊息是否可能描述真實災情，提供人工優先審查排序；模型不能直接決定是否派遣救援。

- Kaggle 教學資料：[Disaster Tweets](https://www.kaggle.com/datasets/vstepanenko/disaster-tweets)（CC0，約 11,000 則）
- 經典題目：[Natural Language Processing with Disaster Tweets](https://www.kaggle.com/competitions/nlp-getting-started/data)
- 本站比較 `TF-IDF + Logistic Regression` 與 `BiLSTM`。簡單 Baseline 贏過 RNN 也完全合理。


## 1. Setup 與固定切分


In [ ]:
%pip -q install kagglehub==1.0.2 gradio==6.20.0


In [ ]:
from pathlib import Path
import random

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

SEED = 20260719
QUICK_MODE = True
MAX_ROWS = 6_000 if QUICK_MODE else None
RNN_EPOCHS = 2 if QUICK_MODE else 5

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
DATASET_HANDLE = "vstepanenko/disaster-tweets"
data_root = Path(kagglehub.dataset_download(DATASET_HANDLE))

data = None
source_csv = None
for csv_path in sorted(data_root.rglob("*.csv")):
    try:
        candidate_frame = pd.read_csv(csv_path)
    except Exception:
        continue
    if {"text", "target"}.issubset(candidate_frame.columns):
        data = candidate_frame[["text", "target"]].dropna().copy()
        source_csv = csv_path
        break
if data is None:
    raise RuntimeError("找不到同時含 text 與 target 的 CSV，請把資料夾列表交給講師。")

data["target"] = data["target"].astype(int)
if MAX_ROWS and len(data) > MAX_ROWS:
    data = data.groupby("target", group_keys=False).apply(
        lambda part: part.sample(min(len(part), MAX_ROWS // 2), random_state=SEED)
    ).reset_index(drop=True)

x_train, x_test, y_train, y_test = train_test_split(
    data["text"].astype(str), data["target"], test_size=0.25,
    stratify=data["target"], random_state=SEED
)
print("Source:", source_csv)
print("Train/Test:", len(x_train), len(x_test))
display(data.head())


## 2. Baseline：TF-IDF + Logistic Regression


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10_000, min_df=2)
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)
baseline = LogisticRegression(max_iter=1_000, random_state=SEED)
baseline.fit(x_train_tfidf, y_train)
baseline_prob = baseline.predict_proba(x_test_tfidf)[:, 1]


## 3. Candidate：Bidirectional LSTM


In [ ]:
MAX_TOKENS = 10_000
SEQUENCE_LENGTH = 40
text_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)
text_vectorizer.adapt(x_train.to_numpy())

rnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    text_vectorizer,
    tf.keras.layers.Embedding(MAX_TOKENS, 32, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="bilstm_candidate")
rnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn.fit(
    x_train.to_numpy(), y_train.to_numpy(),
    validation_split=0.15, epochs=RNN_EPOCHS, batch_size=64, verbose=2
)
rnn_prob = rnn.predict(x_test.to_numpy(), verbose=0).ravel()


## 4. 必做實驗：改分類門檻

災害分流通常更怕漏報（False Negative）。降低門檻可能提高 Recall，但也會增加人工審查量。


In [ ]:
DISASTER_THRESHOLD = 0.35  # TODO(學員必改)：比較 0.30、0.50、0.70

def metric_row(probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    return {
        "precision": precision_score(y_test, predicted, zero_division=0),
        "recall": recall_score(y_test, predicted, zero_division=0),
        "f1": f1_score(y_test, predicted, zero_division=0),
        "review_rate": predicted.mean(),
        "false_negative": int(((y_test.to_numpy() == 1) & (predicted == 0)).sum()),
    }

comparison = pd.DataFrame({
    "TF-IDF baseline": metric_row(baseline_prob, DISASTER_THRESHOLD),
    "BiLSTM candidate": metric_row(rnn_prob, DISASTER_THRESHOLD),
}).T
display(comparison.round(3))


In [ ]:
predicted = (rnn_prob >= DISASTER_THRESHOLD).astype(int)
audit = pd.DataFrame({
    "text": x_test.to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": predicted,
    "risk": rnn_prob,
})
false_positive = audit[(audit.actual == 0) & (audit.predicted == 1)].sort_values("risk", ascending=False)
false_negative = audit[(audit.actual == 1) & (audit.predicted == 0)].sort_values("risk")
print("False Positive examples")
display(false_positive.head(3))
print("False Negative examples")
display(false_negative.head(3))


## 5. 課堂暫時 Demo（選用）


In [ ]:
import gradio as gr

MODEL_VERSION = "station02-bilstm-v1"

def triage_message(text):
    text = (text or "").strip()
    if not text:
        return {"status": "請輸入文字", "model_version": MODEL_VERSION}
    risk = float(rnn.predict(np.array([text]), verbose=0)[0, 0])
    return {
        "risk_score": round(risk, 3),
        "routing": "列入人工優先審查" if risk >= DISASTER_THRESHOLD else "一般訊息佇列",
        "warning": "這是分流建議，不是災情確認或派遣決策。",
        "model_version": MODEL_VERSION,
    }

demo = gr.Interface(
    fn=triage_message,
    inputs=gr.Textbox(lines=3, label="輸入英文短訊息"),
    outputs=gr.JSON(label="分流建議"),
    title="災害訊息分流（課堂暫時 Demo）",
)
print("需要介面時再執行：demo.launch(share=True)")


## 6. 下載迷你實驗卡


In [ ]:
experiment_card = f'''# RNN 站迷你實驗卡

- 組別：請填寫
- 我修改了：DISASTER_THRESHOLD = {DISASTER_THRESHOLD}
- 原本結果：請貼上修改前 Recall / F1 / review rate
- 修改後結果：請貼上修改後 Recall / F1 / review rate
- False Negative 代價：請填寫
- 最大失敗情境：請填寫
- Seed：{SEED}
- 模型版本：{MODEL_VERSION}
- 限制：英文訓練資料不能直接宣稱適用中文災情。
'''
output_path = Path("/content/station02_rnn_experiment_card.md")
output_path.write_text(experiment_card, encoding="utf-8")
print(experiment_card)
print("Saved:", output_path)
